# 01 Exploratory data analysis v2 (cleaned data)

**Owner:** Member 4

**Checklist**

- [x] Price and return plots with the 2020 closure and 2021 to 2023 gap marked (E1)
- [x] ADF and KPSS on log price versus returns, per stock and for the market proxy
- [x] Distribution: histogram, QQ plot, Jarque-Bera, skew, kurtosis
- [x] ACF and PACF of returns with Ljung-Box (never across a gap)
- [x] ACF of squared and absolute returns with ARCH-LM
- [x] Yearly volatility and return table; cross-stock return correlation
- [x] Caption every figure: what we see, what it means for modelling
- [x] Write reports/tables/stat_tests.csv (E3)

Rules: logic lives in `src/`; clear outputs before committing; save figures with `src.viz.style.save`; numbers for the report go to `reports/tables/`.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # run everything from the repo root
sys.path.insert(0, ".")
import numpy as np, pandas as pd
from IPython.display import display
from src.viz.style import apply_style, save
from src.features.build import load_features
from src.eda import plots
apply_style()
feats = load_features()                 # run `python run.py data` first
print(feats.shape, sorted(feats.entity.unique()))

## 1. Price and returns, with the gaps marked

In [ ]:
fig = plots.fig_price_and_returns(feats)
save(fig, "eda_price_returns")

**What we see.** The market proxy (equal-weighted mean of the core stocks) rises and falls in long swings: a deep fall in 2008, a strong rally in 2009 to 2010, a COVID drop in March 2020, and a sharp rise in 2024 to 2025. The red band is the 2020 closure; the grey bands are periods with no data (April to December 2021 and April 2022 to December 2023). Volatility is not constant: the daily-return panel shows calm stretches and bursts of large moves. The very large daily moves in 2003 to 2004 should be read with care, because the proxy starts in October 2003, when only about eight stocks have data.

**What it means for modelling.** Prices wander and do not return to a fixed level, so we forecast returns, not prices. Bursts of volatility call for a volatility model (GARCH). Every rolling window and test must stay inside one unbroken stretch, because no line should join two sides of a gap.

## 2. Stationarity: ADF and KPSS (task E3 table)
Run `python -m src.eda.stat_tests` first. ADF tests the null "unit root", so a small p-value means stationary. KPSS tests the null "stationary", so a small p-value means NOT stationary. Every test is run inside one unbroken stretch with at least 200 days.

In [ ]:
st = pd.read_csv("reports/tables/stat_tests.csv")
mk = st[(st.entity == "MARKET") & st.test.isin(["ADF", "KPSS"])]
display(mk.drop(columns="entity").pivot_table(index=["seg", "start", "end", "n"], columns=["series", "test"],
                                                values="p_value"))

In [ ]:
# How many stock-and-stretch combinations reach each conclusion (returns only)
display(st[st.series == "return"].groupby(["test", "conclusion"]).size().unstack(fill_value=0).T)
display(st[(st.series == "log_price") & st.test.isin(["ADF", "KPSS"])]
        .groupby(["test", "conclusion"]).size().unstack(fill_value=0).T)

**What we see.** Log prices fail the stationarity checks: KPSS rejects stationarity for the market in every long stretch, and ADF mostly cannot reject a unit root. For returns, ADF rejects a unit root almost everywhere, so returns can be modelled as stationary. KPSS still flags some long return stretches, which hints that the mean level of returns or their variance drifts over a decade.

**What it means for modelling.** Work with returns, not prices. Because the series drifts a little over long periods, models should be refitted as new data arrives (the project already refits once per year with an expanding window).

## 3. Distribution of returns

In [ ]:
fig = plots.fig_distribution(feats)
save(fig, "eda_distribution")
display(plots.distribution_summary(feats))

**What we see.** Returns are sharply peaked in the middle with much fatter tails than a normal curve, and the left tail (large falls) is longer than the right. The QQ plot leaves the straight line at both ends. The summary table gives negative skew and a very large excess kurtosis, and the Jarque-Bera test rejects normality.

**What it means for modelling.** Normal errors would badly understate the chance of extreme days. Use Student-t errors in the GARCH model and report errors that are robust to outliers.

## 4. Autocorrelation: ACF and PACF (longest unbroken stretch)

In [ ]:
fig = plots.fig_acf_pacf(feats)
save(fig, "eda_acf_pacf")

**What we see.** Returns are positively autocorrelated at lag 1 and a few more lags (bars outside the shaded band), and the PACF is largest at lag 1. Squared returns are autocorrelated at many lags and the correlation fades slowly.

**What it means for modelling.** A small amount of short-memory structure in the mean supports AR or ARMA models; thin trading and stale prices in a small market are a likely cause (see the limitations file). Slowly fading autocorrelation in squared returns is the classic sign of volatility clustering, which is the case for a GARCH model. Do not read this as profit: the effect is small and may not survive trading costs.

## 5. Autocorrelation and ARCH tests

In [ ]:
rt = st[(st.entity == "MARKET") & (st.series == "return")
        & st.test.isin(["Ljung-Box", "Ljung-Box (squared)", "ARCH-LM", "Jarque-Bera"])]
display(rt.drop(columns=["entity", "series"]))
display(st[(st.series == "return") & st.test.isin(["Ljung-Box", "Ljung-Box (squared)", "ARCH-LM", "Jarque-Bera"])]
        .groupby(["test", "conclusion"]).size().unstack(fill_value=0).T)

**What we see.** The Ljung-Box test finds autocorrelation in returns, and applied to squared returns it finds volatility clustering. The ARCH-LM test finds ARCH effects. For the market proxy these conclusions hold in every long stretch; the short 2020 to 2021 stretch (218 days) shows no clustering. Across all stock-and-stretch combinations, roughly 60 to 70 percent show autocorrelation, volatility clustering or ARCH effects (see the counts above); short stretches have less statistical power, so a "not found" there is weaker evidence.

**What it means for modelling.** Plain linear models are not enough: the variance changes over time, so a model of volatility (GJR-GARCH with Student-t errors) is justified.

## 6. Yearly return and volatility

In [ ]:
yr = plots.yearly_table(feats)
yr.to_csv("reports/tables/yearly_market_stats.csv", index=False)
display(yr)

**What we see.** Annualised volatility is high in the early years and in 2008 to 2009, falls to a low level in 2013 to 2019, jumps in 2020 and again in 2021 to 2022, and is moderate in 2024 to 2025. Years with fewer than about 200 trading days (2003, 2004, 2021, 2022) are partial years because of the gaps in the data, so their totals are not full-year figures.

**What it means for modelling.** Different periods have very different risk, which is why results are reported separately for the validation, COVID and post-gap windows and by regime.

## 7. Cross-stock correlation

In [ ]:
fig, corr = plots.fig_correlation(feats)
save(fig, "eda_correlation")
corr.to_csv("reports/tables/stock_return_correlation.csv")

**What we see.** Every pair of stocks is positively correlated, at moderate levels (roughly 0.2 to 0.45), with banks and diversified groups (for example NDB, DFCC, JKH, SAMP) moving together most and DIAL and LIOC the least.

**What it means for modelling.** A common market factor drives part of every stock's return, which is why the panel models use the same-day market return as a feature. Stocks are not independent, so pooled results should not be treated as thirteen independent experiments.